In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

from collections import defaultdict
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys.
    """
    rng = np.random.default_rng(seed)
    transactions: list[set[str]] = []
    for _ in range(n):
        basket: set[str] = set()
        for bundle_items, prob in BUNDLES:
            if rng.random() < prob:
                for item in bundle_items:
                    if rng.random() < 0.85:
                        basket.add(item)
        n_random = rng.poisson(2)
        if n_random > 0:
            random_items = rng.choice(
                PRODUCTS, size=int(min(n_random, 5)), replace=False
            )
            basket.update(random_items)
        if basket:
            transactions.append(basket)
    return transactions


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. Used as input to mlxtend FP-Growth (via .to_pandas()).
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.2: FP-Growth via mlxtend
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Use mlxtend's FP-Growth implementation end-to-end
#   - Convert polars transactions into the one-hot format FP-Growth expects
#   - Compare Apriori and FP-Growth on the same min_support
#   - Verify that both algorithms produce the same frequent itemsets
#
# PREREQUISITES:
#   - 01_apriori_from_scratch.py
#
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Theory — FP-tree construction and recursive mining
#   2. Build — wrap mlxtend FP-Growth in a polars-friendly call
#   3. Train — run FP-Growth on 2,500 SG retail baskets
#   4. Visualise — Apriori vs FP-Growth itemset overlap
#   5. Apply — GrabFood order-bundle mining at city scale
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict

import polars as pl


# mlxtend requires pandas internally. Use at the boundary only — keep
# the rest of this file polars-native.
from mlxtend.frequent_patterns import association_rules as mlx_association_rules
from mlxtend.frequent_patterns import fpgrowth as mlx_fpgrowth



## THEORY — FP-Tree and Recursive Mining


In [ ]:
# FP-Growth builds a prefix tree (FP-tree) in TWO passes over the data,
# then mines frequent itemsets recursively from conditional pattern
# bases in the tree — no more DB scans after tree construction, and no
# candidate generation at all.
#
# Guarantees: same frequent itemsets as Apriori given the same
# min_support, typically 2-10x faster on dense data, 10-100x faster on
# large data. Downside: the FP-tree has to fit in memory.



## TASK 2 — BUILD: polars-friendly FP-Growth wrapper


Returns ``(frequent_itemsets_df, rules_df)`` as polars DataFrames.


In [ ]:
def run_fp_growth(
    transactions: list[set[str]],
    min_support: float,
    min_confidence: float = 0.3,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    # TODO: build the one-hot transaction matrix using
    # transactions_to_onehot() and then convert to pandas for mlxtend.
    # Hint: polars DataFrames have a .to_pandas() method.
    onehot_pl = ____
    onehot_pd = ____

    # TODO: call mlx_fpgrowth() with use_colnames=True so the returned
    # frame uses product names (not column indices).
    fp_frequent = ____

    # TODO: derive association rules from `fp_frequent`. Filter on
    # confidence with min_threshold=min_confidence.
    # Hint: mlx_association_rules(fp_frequent, metric="confidence", ...)
    fp_rules = ____

    frequent_pl = pl.DataFrame(
        {
            "itemset": [
                format_itemset(items) for items in fp_frequent["itemsets"].tolist()
            ],
            "size": [len(items) for items in fp_frequent["itemsets"].tolist()],
            "support": fp_frequent["support"].astype(float).tolist(),
        }
    )
    rules_pl = pl.DataFrame(
        {
            "antecedent": [
                format_itemset(items) for items in fp_rules["antecedents"].tolist()
            ],
            "consequent": [
                format_itemset(items) for items in fp_rules["consequents"].tolist()
            ],
            "support": fp_rules["support"].astype(float).tolist(),
            "confidence": fp_rules["confidence"].astype(float).tolist(),
            "lift": fp_rules["lift"].astype(float).tolist(),
        }
    )
    return frequent_pl, rules_pl



## TASK 3 — TRAIN: run FP-Growth on the same baskets used in 5.1


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.3

print("\n=== FP-Growth (mlxtend) ===")
fp_frequent_df, fp_rules_df = run_fp_growth(
    transactions, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE
)
print(f"  Frequent itemsets: {fp_frequent_df.height}")
print(f"  Association rules: {fp_rules_df.height}")



## TASK 4 — VISUALISE: compare Apriori and FP-Growth


Minimal Apriori — same contract as 01_apriori_from_scratch.apriori().


In [ ]:
def _apriori(
    transactions: list[set[str]], min_support: float
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1
    freq: dict[frozenset[str], float] = {}
    level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq[fs] = count / n
            level.append(fs)
    k = 2
    while level:
        prev_set = set(level)
        candidates: set[frozenset[str]] = set()
        for i, a in enumerate(level):
            for b in level[i + 1 :]:
                u = a | b
                if len(u) == k and all((u - frozenset([it])) in prev_set for it in u):
                    candidates.add(u)
        if not candidates:
            break
        counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            tf = frozenset(txn)
            for c in candidates:
                if c.issubset(tf):
                    counts[c] += 1
        level = []
        for c, ct in counts.items():
            if ct >= min_count:
                freq[c] = ct / n
                level.append(c)
        k += 1
    return freq


apriori_itemsets = _apriori(transactions, min_support=MIN_SUPPORT)
apriori_set = {format_itemset(s) for s in apriori_itemsets}
fp_set = set(fp_frequent_df["itemset"].to_list())

print("\n=== Apriori vs FP-Growth ===")
print(f"  Apriori   itemsets: {len(apriori_set)}")
print(f"  FP-Growth itemsets: {len(fp_set)}")
print(f"  Intersection:       {len(apriori_set & fp_set)}")

# TODO: compute the Jaccard agreement between the two sets. Remember to
# guard against divide-by-zero on an empty union.
# Hint: |A ∩ B| / |A ∪ B|
agreement = ____
print(f"  Jaccard agreement:  {agreement:.2%}")



### Checkpoint


In [ ]:
assert fp_frequent_df.height > 0, "FP-Growth should find at least one itemset"
assert fp_rules_df.height > 0, "FP-Growth should generate at least one rule"
assert (
    agreement >= 0.90
), f"Apriori and FP-Growth should agree on >=90% of itemsets; got {agreement:.2%}"
print("\n[ok] Checkpoint passed — FP-Growth matches Apriori on frequent itemsets\n")

fp_frequent_df.write_csv(OUTPUT_DIR / "fp_growth_itemsets.csv")
fp_rules_df.write_csv(OUTPUT_DIR / "fp_growth_rules.csv")



## TASK 5 — APPLY: GrabFood order-bundle mining


In [ ]:
# SCENARIO: GrabFood SG processes ~800K orders/day across 12K+ merchants.
# Weekly mining run (~5.6M transactions) to surface high-lift menu pairs
# for merchant bundle promotions. FP-Growth's single-pass tree is 30-60x
# faster than Apriori on this shape of data.
#
# BUSINESS IMPACT: 10% AOV lift on S$18 average order with 800K orders/day
# is ~S$1.4M/day GMV — S$45M/month for ~S$50 of compute per weekly run.



## REFLECTION


[x] Wrapped mlxtend FP-Growth in a polars-friendly call boundary
  [x] Converted basket-sets to one-hot for FP-Growth input
  [x] Verified Apriori and FP-Growth agree on frequent itemsets

  Next: 03_rule_evaluation.py — turn frequent itemsets into association
  rules with support, confidence, lift, and conviction.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

